# EM-11 model comparison, notebook 2 of 2: score and compare

Scores every analysis window with YAMNet (CPU, as on the server) and AST (T4 GPUs), times each
model, re-runs the original 280-clip evaluation as a sanity check, then writes the comparison:
`results/report.md` and `results/report.json`.

**Settings (right-hand panel)**

- Accelerator: **GPU T4 x2**.
- Internet: **On** (code from GitHub, packages from PyPI, YAMNet from TF Hub, AST from Hugging Face).
- Input: **Add Input → Your Work → notebook 1's output** (the version you want). Do not attach the
  FSD50K dataset here.
- To continue a run that did not complete: also attach **that run's output** (this notebook's
  earlier version). Its finished score files are reused and their stages skipped.
- Environment: *Pin to original environment*.

Run it with **Save Version → Save & Run All (Commit)**.

**Preflight.** Before any long stage, every stage runs on a small set of clips that includes the ones
most likely to break something (the shortest, the longest, those with the shortest last AST piece).
It takes about 10 minutes. When the log shows `PREFLIGHT PASSED`, the long stages start. If
something is wrong in the code, it stops here, not hours later.

**After the preflight, the notebook always finishes.** Kaggle keeps no output of a version that ends
in an error, so a stage that fails later is recorded and the stages that do not need it go on. The
last cell then says `RUN INCOMPLETE` and lists what failed; every finished score file is in `banks/`
of that version's output, ready to be reused. Do not stop a run after the preflight: a stopped
version may keep nothing either.

With notebook 1's smoke output the whole run takes about 30 minutes; with the full output, about
4 hours (less when an earlier run's scores are reused). Send back `results/report.md`,
`results/report.json` and `results/verify-report.md`.

In [ ]:
REF = "main"    # the code to run: a branch, tag or commit ("main" has every fix)
SAMPLE = 2000   # windows AST also scores in half precision, to check that half precision changes nothing
RESUME = True   # reuse score files of an earlier run of this notebook attached as input (same prepared data)

In [ ]:
import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from IPython.display import Markdown, display


def sh(*cmd, check=True, cwd=None, tail=None):
    """Run one command, its output shown here (a child's own output would not reach the notebook).
    With ``tail``, only its last lines are shown, unless it fails: then all of them."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        if tail:
            lines.append(line)
        else:
            print(line, end="", flush=True)
    failed = proc.wait()
    if tail:
        print("".join(lines if failed else lines[-tail:]), end="", flush=True)
    if failed and check:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc.returncode


def out(*cmd, cwd=None):
    return subprocess.check_output([str(c) for c in cmd], text=True, cwd=cwd).strip()


def find(name, root=Path("/kaggle/input"), depth=5, dirs=False):
    """Paths under /kaggle/input whose name contains ``name``, nearest first."""
    hits = [p for d in range(1, depth + 1) for p in root.glob("/".join("*" * d))
            if name in p.name.lower() and (p.is_dir() or not dirs)]
    return sorted(hits, key=lambda p: len(p.parts))


UV = [sys.executable, "-m", "uv"]


def venv(name, requirements):
    """A clean Python 3.12 environment with pinned packages; what it holds is saved next to the results."""
    path = Path(f"/tmp/venv-{name}")
    shutil.rmtree(path, ignore_errors=True)
    sh(*UV, "venv", "--quiet", "--python", "3.12", path)
    sh(*UV, "pip", "install", "--quiet", "--python", path / "bin/python", "-r", CODE / requirements)
    freeze = out(*UV, "pip", "freeze", "--python", path / "bin/python")
    (WORK / f"env-{name}.txt").write_text(freeze + "\n")
    return path / "bin/python"

WORK = Path("/kaggle/working")
CODE, DATA, WEIGHTS = Path("/tmp/encoremoment"), Path("/tmp/em11-data"), Path("/tmp/ast")
BANKS, TIMING, RESULTS, LOGS = (WORK / d for d in ("banks", "timing", "results", "logs"))
for d in (BANKS, TIMING, RESULTS, LOGS):
    d.mkdir(parents=True, exist_ok=True)
os.environ["TFHUB_CACHE_DIR"] = "/tmp/tfhub"
TOOL = CODE / "server/tools/em11_compare.py"
ALL = "curated,mixed,dev,everyday"
PHASES = {}
FAILED = {}  # stage -> why: what failed (or could not run) after the preflight


def parallel(phase, jobs, every=600, strict=False):
    """Run (name, command, output[, needs]) jobs at once, each logging to logs/NAME.log.

    A job whose output already exists (kept from an earlier run) is skipped, and so is one whose
    ``needs`` (files other jobs write) are missing. A failed job is recorded in FAILED and the others
    go on, so the notebook still finishes and Kaggle keeps every finished score file. With ``strict``
    (the preflight) a failure stops the other jobs and the notebook."""
    started, running = time.time(), {}
    for name, cmd, output, *needs in jobs:
        if Path(output).exists():
            print(f"{name}: kept from an earlier run", flush=True)
            continue
        missing = [Path(f).name for f in (needs[0] if needs else []) if not Path(f).exists()]
        if missing:
            FAILED[name] = f"not run: needs {', '.join(missing)}"
            print(f"{name}: {FAILED[name]}", flush=True)
            continue
        log = open(LOGS / f"{name}.log", "w")
        running[name] = (subprocess.Popen([str(c) for c in cmd], stdout=log, stderr=subprocess.STDOUT, cwd=CODE), log)
        print("started", name, flush=True)
    last = time.time()
    while running:
        time.sleep(10)
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            lines = (LOGS / f"{name}.log").read_text().strip().splitlines()
            print(f"{name}: exit {proc.returncode} {lines[-1:]}", flush=True)
            if proc.returncode:
                print("\n".join(lines[-30:]))
                if strict:
                    for p, _ in running.values():
                        p.terminate()
                    raise RuntimeError(f"{name} failed: see logs/{name}.log (the last lines are above)")
                FAILED[name] = f"exit {proc.returncode}: {lines[-1] if lines else ''} (logs/{name}.log)"
                print(f"STAGE FAILED: {name} (the last lines are above); the other stages go on", flush=True)
        if running and time.time() - last > every:
            last = time.time()
            for name in running:
                lines = (LOGS / f"{name}.log").read_text().strip().splitlines()
                print(f"  {name}: {lines[-1] if lines else '...'}", flush=True)
    PHASES[phase] = round((time.time() - started) / 60, 1)
    print(f"{phase}: {PHASES[phase]} minutes", flush=True)


def score(py, variant, out, sets, *more):
    """A score job: (name, command, output file)."""
    return (Path(out).stem, [py, TOOL, "score", "--data", DATA, "--model", variant, "--sets", sets, "--out", out, *more], out)


def ast(variant, out, sets, gpu, shards=None, precision="fp32", *more):
    """An AST score job on GPU ``gpu``, scoring share ``gpu`` of ``shards`` (default: one share per GPU)."""
    shards = shards or GPUS
    return score(AST, variant, out, sets, "--weights", WEIGHTS, "--device", f"cuda:{gpu % GPUS}", "--precision", precision,
                 "--shard", f"{gpu % shards}/{shards}", "--threads", 1, *more)


def guarded(name, *cmd, tail=1):
    """Run one command after the preflight: a failure is recorded in FAILED and the notebook goes on."""
    try:
        sh(*cmd, cwd=CODE, tail=tail)
        return True
    except Exception as e:  # noqa: BLE001 - recorded and reported in the last cell
        FAILED[name] = f"{type(e).__name__}: {e}"
        print(f"STAGE FAILED: {name}; the other stages go on", flush=True)
        return False


def timing(py, variant, out, device, *more):
    return [py, TOOL, "timing", "--data", DATA, "--model", variant, "--device", device, "--out", out, *more]


def banks(folder, *patterns, shards=None):
    return ",".join(str(folder / p.format(g)) for p in patterns for g in range(shards or GPUS))

## Data from notebook 1, code and environment

In [ ]:
tars = find("em11-data.tar")
if len(tars) != 1:
    raise SystemExit(f"Add Input: notebook 1's output, one version only (found {tars})")
shutil.rmtree(DATA, ignore_errors=True)
sh("tar", "-xf", tars[0], "-C", DATA.parent)
manifest = json.loads((DATA / "manifest.json").read_text())
code = json.loads((DATA / "code.json").read_text())
SMOKE = manifest["limit"] is not None
print(f"data from {tars[0]}: {len(manifest['files'])} clips, created {manifest['created']}, "
      f"copy used: {manifest['checks']['copy_used']}, smoke: {SMOKE}")

shutil.rmtree(CODE, ignore_errors=True)
sh("git", "clone", "--quiet", code["repo"], CODE)
sh("git", "-C", CODE, "checkout", "--quiet", REF or code["commit"])
COMMIT = out("git", "-C", CODE, "rev-parse", "HEAD")
print("code at", COMMIT, "(notebook 1 ran", code["commit"] + ")")

sh(sys.executable, "-m", "pip", "install", "--quiet", "uv==0.9.5")
TF = venv("tf", "server/requirements-yamnet.txt")
AST = venv("ast", "server/tools/kaggle/requirements-ast.txt")

## Score files of an earlier run

With `RESUME = True`, score files (`banks/*.npz`) of an earlier run of this notebook that is attached
as input are reused when they were made from the same prepared data; their stages are skipped.

In [ ]:
import numpy as np  # the notebook's own numpy, only to read score-file metadata

REUSED = []
for folder in (find("banks", dirs=True, depth=5) if RESUME else []):
    for f in sorted(folder.glob("*.npz")):
        with np.load(f) as z:
            meta = json.loads(str(z["meta"]))
        if meta.get("data_created") != manifest["created"] or meta.get("limit") or meta.get("selection"):
            print(f"not reused (made from other data): {f}")
        elif not (BANKS / f.name).exists():
            shutil.copy(f, BANKS / f.name)
            REUSED.append(f.name)
            print(f"reused {f.name}: {meta['rows']} rows, {meta['variant']} {meta['precision']}, code {meta.get('commit')}")
print("reused:", REUSED or "nothing")

## GPUs and model weights (pinned, checked)

In [ ]:
sh("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")
GPUS = int(out(AST, "-c", "import torch; print(torch.cuda.device_count())"))
if GPUS < 1:
    raise SystemExit("Accelerator: GPU T4 x2")
print(GPUS, "GPUs,", os.cpu_count(), "CPUs")

PINS = json.loads(out(AST, "-c", "import json, sys; sys.path.insert(0, 'server/tools'); import em11_compare; "
                      "print(json.dumps(em11_compare.PINS))", cwd=CODE))
pin = PINS["ast"]
sh(AST, "-c", "from huggingface_hub import snapshot_download; "
   f"snapshot_download({pin['repo']!r}, revision={pin['revision']!r}, local_dir={str(WEIGHTS)!r}, "
   f"allow_patterns=['*.json', {pin['file']!r}])")
digest = out("sha256sum", WEIGHTS / pin["file"]).split()[0]
assert digest == pin["sha256"], f"AST weights {digest}, expected {pin['sha256']}"
print("AST", pin["repo"], pin["revision"][:12], "sha256 ok")
# YAMNet from TF Hub; the score stage checks its digest against the pin.
sh(TF, "-c", "import sys; sys.path.insert(0, 'server'); from encore_server.models import YamnetModel; "
   "print(YamnetModel().name)", cwd=CODE)

## Preflight: every stage on the clips most likely to break one

Every score job of the full run (each model, each precision, whole clips), the timing paths and the
whole analysis, on one clip per group plus the shortest and longest clips and those with the
shortest last 10.24 s piece. Nothing here is kept; it only has to pass.

In [ ]:
started = time.time()
PRE = Path("/tmp/preflight")
shutil.rmtree(PRE, ignore_errors=True)
for d in ("banks", "timing", "results"):
    (PRE / d).mkdir(parents=True)
SEL = PRE / "selection.json"
sh(AST, TOOL, "preflight", "--data", DATA, "--out", SEL, cwd=CODE)
only = ("--selection", SEL)
B = PRE / "banks"
pre = lambda jobs: [(f"preflight-{name}", cmd, output) for name, cmd, output in jobs]  # own log names
parallel("preflight: yamnet, ast-pad10s fp32, ast-cut1s", strict=True, jobs=pre([
    score(TF, "yamnet", B / "yamnet.npz", ALL, "--threads", 1, *only),
    ast("ast-pad10s", B / "pad10s-fp32.npz", "curated,mixed,dev", 0, 1, "fp32", *only),
    ast("ast-cut1s", B / "cut1s.npz", ALL, 1, 1, "fp32", *only)]))
parallel("preflight: ast-pad10s fp16, ast-clip", strict=True, jobs=pre([
    ast("ast-pad10s", B / "pad10s-fp16.npz", "everyday", 0, 1, "fp16", "--skip", B / "pad10s-fp32.npz", *only),
    ast("ast-clip", B / "clip.npz", "curated,dev,everyday", 1, 1, "fp32", *only),
    ast("ast-pad10s", B / "pad10s-fp16-sample.npz", "curated", 0, 1, "fp16", "--sample", 50, *only)]))
for cmd in (timing(TF, "yamnet", PRE / "timing/yamnet-cpu2.json", "cpu", "--threads", 2, "--windows", 2),
            timing(AST, "ast-pad10s", PRE / "timing/pad10s-t4-fp16.json", "cuda:0", "--weights", WEIGHTS,
                   "--precision", "fp16", "--windows", 2),
            timing(AST, "ast-cut1s", PRE / "timing/cut1s-cpu2.json", "cpu", "--weights", WEIGHTS, "--threads", 2,
                   "--windows", 2)):
    sh(*cmd, cwd=CODE, tail=1)
sh(AST, TOOL, "analyze", "--data", DATA, "--selection", SEL, "--out", PRE / "results", "--workers", 2,
   "--bank", f"yamnet={B / 'yamnet.npz'}", "--bank", f"ast-pad10s={B / 'pad10s-fp32.npz'},{B / 'pad10s-fp16.npz'}",
   "--bank", f"ast-cut1s={B / 'cut1s.npz'}", "--bank", f"ast-clip={B / 'clip.npz'}",
   "--check", f"ast-pad10s={B / 'pad10s-fp16-sample.npz'}",
   *[a for p in sorted((PRE / "timing").glob("*.json")) for a in ("--timing", p)], cwd=CODE, tail=8)
PHASES["preflight"] = round((time.time() - started) / 60, 1)
print(f"PREFLIGHT PASSED in {PHASES['preflight']} minutes: the long stages start now.")

## Score

Each model's scores for every distinct window go into `banks/` (kept with this version, so a later
run can reuse them). Cheap stages come first. A failed job is recorded and the rest go on. AST jobs split the windows between the GPUs; YAMNet
runs on one CPU thread beside them, and the sanity check (`em11_eval.py` on the original 280 clips,
CPU) beside the first stage. Half precision is used only for AST on the everyday clips; a sample is
scored both ways and compared.

In [ ]:
first = [ast("ast-clip", BANKS / f"clip-{g}.npz", "curated,dev,everyday", g) for g in range(GPUS)]
if not SMOKE:  # the 280 original clips are only prepared in a full run
    first.append(("sanity", [TF, TOOL, "sanity", "--data", DATA, "--work", "/tmp/sanity", "--out", RESULTS / "sanity.json"],
                  RESULTS / "sanity.json"))
parallel("ast-clip + sanity", first)
parallel("ast-cut1s", [ast("ast-cut1s", BANKS / f"cut1s-{g}.npz", ALL, g) for g in range(GPUS)])
parallel("yamnet + ast-pad10s fp32",
         [score(TF, "yamnet", BANKS / "yamnet.npz", ALL, "--threads", 1)]
         + [ast("ast-pad10s", BANKS / f"pad10s-fp32-{g}.npz", "curated,mixed,dev", g) for g in range(GPUS)])
fp32 = banks(BANKS, "pad10s-fp32-{}.npz").split(",")  # their windows are not scored again in half precision
skip = [a for f in fp32 for a in ("--skip", f)]
parallel("ast-pad10s fp16 everyday",
         [(*ast("ast-pad10s", BANKS / f"pad10s-fp16-{g}.npz", "everyday", g, None, "fp16", *skip), fp32)
          for g in range(GPUS)])
parallel("fp16 sample", [ast("ast-pad10s", BANKS / "pad10s-fp16-sample.npz", "curated", 0, 1, "fp16", "--sample", SAMPLE)])

## Time per window

One window at a time, nothing else running: YAMNet on 2 CPU threads (the cloud server) and on all;
AST on a T4 and on 2 CPU threads.

In [ ]:
started = time.time()
for cmd in (timing(TF, "yamnet", TIMING / "yamnet-cpu2.json", "cpu", "--threads", 2),
            timing(TF, "yamnet", TIMING / "yamnet-cpu.json", "cpu"),
            timing(AST, "ast-pad10s", TIMING / "pad10s-t4.json", "cuda:0", "--weights", WEIGHTS),
            timing(AST, "ast-pad10s", TIMING / "pad10s-t4-fp16.json", "cuda:0", "--weights", WEIGHTS, "--precision", "fp16"),
            timing(AST, "ast-cut1s", TIMING / "cut1s-t4.json", "cuda:0", "--weights", WEIGHTS),
            timing(AST, "ast-cut1s", TIMING / "cut1s-cpu2.json", "cpu", "--weights", WEIGHTS, "--threads", 2),
            timing(AST, "ast-pad10s", TIMING / "pad10s-cpu2.json", "cpu", "--weights", WEIGHTS, "--threads", 2, "--windows", 20)):
    guarded("timing " + Path(cmd[cmd.index("--out") + 1]).stem, *cmd)
PHASES["timing"] = round((time.time() - started) / 60, 1)

## Compare

In [ ]:
started = time.time()
parts = {"yamnet": banks(BANKS, "yamnet.npz", shards=1),
         "ast-pad10s": banks(BANKS, "pad10s-fp32-{}.npz", "pad10s-fp16-{}.npz"),  # full precision first
         "ast-cut1s": banks(BANKS, "cut1s-{}.npz"),
         "ast-clip": banks(BANKS, "clip-{}.npz")}
check = BANKS / "pad10s-fp16-sample.npz"
missing = [Path(f).name for files in parts.values() for f in files.split(",") if not Path(f).exists()]
missing += [] if check.exists() else [check.name]
if missing:
    FAILED["analyze"] = f"not run: needs {', '.join(missing)}"
    print("analyze", FAILED["analyze"])
else:
    args = [a for name, files in parts.items() for a in ("--bank", f"{name}={files}")]
    args += ["--check", f"ast-pad10s={check}", *[a for p in sorted(TIMING.glob("*.json")) for a in ("--timing", p)]]
    if (RESULTS / "sanity.json").exists():
        args += ["--sanity", RESULTS / "sanity.json"]
    guarded("analyze", AST, TOOL, "analyze", "--data", DATA, "--out", RESULTS, "--commit", COMMIT,
            "--workers", os.cpu_count(), *args, tail=3)
PHASES["analyze"] = round((time.time() - started) / 60, 1)
shutil.copy(DATA / "verify-report.md", RESULTS / "verify-report.md")
for f in WORK.glob("env-*.txt"):
    shutil.move(f, RESULTS / f.name)
if "analyze" not in FAILED:
    display(Markdown((RESULTS / "report.md").read_text()))

## Run summary (minutes per phase; windows per second per score file)

In [ ]:
print(json.dumps(PHASES, indent=1))
print("reused from an earlier run:", REUSED or "nothing")
for f in sorted(BANKS.glob("*.npz")):
    try:
        with np.load(f) as z:
            m = json.loads(str(z["meta"]))
        print(f"{f.name}: {m['rows']} rows, {m['per_second']} per second, {m['wall_seconds']} s")
    except Exception as e:  # noqa: BLE001 - reported below; the notebook must still finish
        FAILED[f.name] = f"unreadable: {e}"
if SMOKE:
    print("A full run scores about 335,000 distinct windows per window model; divide by the rates above "
          "(both GPUs together) for a rough time.")
(RESULTS / "run.json").write_text(json.dumps({"commit": COMMIT, "data": code, "phases": PHASES, "gpus": GPUS,
                                              "reused": REUSED, "complete": not FAILED, "failed": FAILED},
                                             indent=1) + "\n")
if FAILED:
    print("\nRUN INCOMPLETE. Failed or not run:")
    for name, why in FAILED.items():
        print(f"  {name}: {why}")
    print("Every finished score file is kept in banks/ of this version's output. Fix the cause, then run "
          "again with this version's output attached as input: those files are reused and their stages skipped.")
else:
    print("\nRUN COMPLETE. Send back results/report.md, results/report.json and results/verify-report.md.")